# Group 6 - Supply Chain Disruption Response Coordinator

| | |
|---|---|
| **Student** | Arpita Mishra |
| **Course** | CSE476 Agentic AI and Intelligent Automation |
| **Instructor** | Rohit Bharti |
| **Institution** | School of Computer Science and Engineering, Lovely Professional University |

## What this notebook shows
A **blackboard** multi-agent system built on **Microsoft Semantic Kernel**. For each disruption report:

1. `MonitorAgent` parses the report and posts `disruption`, `affected_items`, `affected_sites`.
2. `InventoryAgent`, `SupplierAgent` and `RouteAgent` run **in parallel** and post stock, supplier and route signals.
3. `CostAgent` (optional extension beyond the required five agents) prices every option.
4. `CoordinatorAgent` reads **only** the blackboard and decides **reorder / reroute / hold** and who to notify, or **holds and escalates to a human** if a signal is missing.

Every line of the trace below comes from a real run. Tool calls are indented; blackboard posts are not.

In [1]:
import os, sys, json
sys.path.insert(0, os.getcwd())
from config import load_settings
from orchestrator import run_disruption, REPORTS_DIR

settings = load_settings()
print(f"Model lane: mode={settings.mode}, provider={settings.provider}, model={settings.model}")
print("Reports:", [p.name for p in sorted(REPORTS_DIR.glob('*.json'))])

Model lane: mode=offline, provider=offline, model=rule-based
Reports: ['01_supplier_delay.json', '02_stockout.json', '03_route_block.json', '04_missing_data.json', '05_conflicting.json', '06_false_alarm_hold.json']


## Scenario 1 - Supplier delay (expected: reorder from a faster alternative supplier)

In [2]:
r1 = await run_disruption(REPORTS_DIR / "01_supplier_delay.json")

RUN 20261111-195647-c2d2e2 | report DR-2026-1101 (supplier_delay) | mode=offline
-- step 1: MonitorAgent --
[MonitorAgent] reading report DR-2026-1101: "SUPPLIER DELAY: Apex Components (SUP-01) informs that the shipment of ITM-001 (4..."
    [MonitorAgent] tool parse_disruption(report_text=SUPPLIER DELAY: Apex Components (SUP-...) -> {'type': 'supplier_delay', 'delay_days': 10, 'needed_by_days': 7, 'disrupted_supplier': 'SUP-01', 'blocked_...
    [MonitorAgent] tool extract_items_and_sites(report_text=SUPPLIER DELAY: Apex Components (SUP-...) -> {'items': ['ITM-001'], 'sites': ['WH-DEL'], 'unknown_items': [], 'unknown_sites': []}
[MonitorAgent] posted 'disruption' to blackboard (supplier_delay reported; delay 10d, needed by 7d)
[MonitorAgent] posted 'affected_items' to blackboard (ITM-001)
[MonitorAgent] posted 'affected_sites' to blackboard (WH-DEL)
-- step 2: InventoryAgent | SupplierAgent | RouteAgent (parallel) --
    [InventoryAgent] tool get_stock_level(item_id=ITM-001, site_id=W

## Scenario 2 - Stockout (expected: reorder from an alternative supplier, customer service warned)

In [3]:
r2 = await run_disruption(REPORTS_DIR / "02_stockout.json")

RUN 20261111-195647-917da3 | report DR-2026-1102 (stockout) | mode=offline
-- step 1: MonitorAgent --
[MonitorAgent] reading report DR-2026-1102: "STOCKOUT ALERT: ITM-004 (aluminium frames) has hit zero on-hand at WH-MUM after ..."
    [MonitorAgent] tool parse_disruption(report_text=STOCKOUT ALERT: ITM-004 (aluminium fr...) -> {'type': 'stockout', 'delay_days': 0, 'needed_by_days': 6, 'disrupted_supplier': None, 'blocked_route': None}
    [MonitorAgent] tool extract_items_and_sites(report_text=STOCKOUT ALERT: ITM-004 (aluminium fr...) -> {'items': ['ITM-004'], 'sites': ['WH-MUM'], 'unknown_items': [], 'unknown_sites': []}
[MonitorAgent] posted 'disruption' to blackboard (stockout reported; delay 0d, needed by 6d)
[MonitorAgent] posted 'affected_items' to blackboard (ITM-004)
[MonitorAgent] posted 'affected_sites' to blackboard (WH-MUM)
-- step 2: InventoryAgent | SupplierAgent | RouteAgent (parallel) --
    [InventoryAgent] tool get_stock_level(item_id=ITM-004, site_id=WH-MUM) -> {'fo

## Scenario 3 - Route block (expected: reroute over an open lane)

In [4]:
r3 = await run_disruption(REPORTS_DIR / "03_route_block.json")

RUN 20261111-195647-c473ae | report DR-2026-1103 (route_block) | mode=offline


-- step 1: MonitorAgent --
[MonitorAgent] reading report DR-2026-1103: "ROUTE BLOCKED: Lane RT-03 (NH-16 coastal road) into WH-KOL is closed by flooding..."
    [MonitorAgent] tool parse_disruption(report_text=ROUTE BLOCKED: Lane RT-03 (NH-16 coas...) -> {'type': 'route_block', 'delay_days': 8, 'needed_by_days': 5, 'disrupted_supplier': None, 'blocked_route': ...
    [MonitorAgent] tool extract_items_and_sites(report_text=ROUTE BLOCKED: Lane RT-03 (NH-16 coas...) -> {'items': ['ITM-006'], 'sites': ['WH-KOL'], 'unknown_items': [], 'unknown_sites': []}
[MonitorAgent] posted 'disruption' to blackboard (route_block reported; delay 8d, needed by 5d)
[MonitorAgent] posted 'affected_items' to blackboard (ITM-006)
[MonitorAgent] posted 'affected_sites' to blackboard (WH-KOL)
-- step 2: InventoryAgent | SupplierAgent | RouteAgent (parallel) --
    [InventoryAgent] tool get_stock_level(item_id=ITM-006, site_id=WH-KOL) -> {'found': True, 'item_id': 'ITM-006', 'site_id': 'WH-KOL', 'on_hand': 70}
 

    [RouteAgent] tool get_route_delay(route_id=RT-04) -> {'found': True, 'route_id': 'RT-04', 'status': 'open', 'transit_days': 1, 'current_delay_days': 0, 'effecti...
[RouteAgent] posted 'route_options:WH-KOL' to blackboard (RT-03 blocked; open alternatives: RT-04 air 1d)
-- step 3: CostAgent --
    [CostAgent] tool estimate_hold_cost(item_id=ITM-006, shortfall_units=106) -> {'option': 'hold', 'shortfall_units': 106, 'cost': 1590.0}
    [CostAgent] tool estimate_reroute_cost(route_id=RT-04, qty=130) -> {'option': 'reroute', 'route_id': 'RT-04', 'qty': 130, 'cost': 2790.0}
[CostAgent] posted 'cost_options:ITM-006@WH-KOL' to blackboard (hold  $1,590; reroute RT-04 $2,790)
-- step 4: CoordinatorAgent --
    [CoordinatorAgent] tool check_required_signals(postings={'disruption': {'status': 'ok', 'valu..., required_keys=['disruption', 'affected_items', 'aff...) -> {'complete': True, 'missing': [], 'malformed': []}
    [CoordinatorAgent] tool assess_options(disruption={'type': 'route_block',

## Scenario 4 - Honest failure: missing data (expected: HOLD + ESCALATE TO HUMAN)
The stock record for ITM-008 at WH-BLR does not exist (simulated WMS sync failure) and ITM-008 is sole-sourced.
The InventoryAgent posts a **missing** signal instead of guessing, and the Coordinator refuses to decide.

In [5]:
r4 = await run_disruption(REPORTS_DIR / "04_missing_data.json")

RUN 20261111-195647-8a01e8 | report DR-2026-1104 (missing_data_failure) | mode=offline
-- step 1: MonitorAgent --
[MonitorAgent] reading report DR-2026-1104: "STOCKOUT: planner at WH-BLR says firmware chips ITM-008 are running out and the ..."
    [MonitorAgent] tool parse_disruption(report_text=STOCKOUT: planner at WH-BLR says firm...) -> {'type': 'stockout', 'delay_days': 0, 'needed_by_days': 3, 'disrupted_supplier': None, 'blocked_route': None}
    [MonitorAgent] tool extract_items_and_sites(report_text=STOCKOUT: planner at WH-BLR says firm...) -> {'items': ['ITM-008'], 'sites': ['WH-BLR'], 'unknown_items': [], 'unknown_sites': []}
[MonitorAgent] posted 'disruption' to blackboard (stockout reported; delay 0d, needed by 3d)
[MonitorAgent] posted 'affected_items' to blackboard (ITM-008)
[MonitorAgent] posted 'affected_sites' to blackboard (WH-BLR)
-- step 2: InventoryAgent | SupplierAgent | RouteAgent (parallel) --
    [InventoryAgent] tool get_stock_level(item_id=ITM-008, site_id=WH-

## Scenario 5 - Conflicting options (expected: cheapest *viable* option wins)
Procurement wants to reorder, logistics wants to reroute. Both are viable; *hold* is the cheapest on paper but not viable.

In [6]:
r5 = await run_disruption(REPORTS_DIR / "05_conflicting.json")

RUN 20261111-195647-7744a6 | report DR-2026-1105 (conflicting_options) | mode=offline
-- step 1: MonitorAgent --
[MonitorAgent] reading report DR-2026-1105: "ROUTE BLOCKED: Western Roadways reports a landslide on lane RT-05 (NH-48) into W..."
    [MonitorAgent] tool parse_disruption(report_text=ROUTE BLOCKED: Western Roadways repor...) -> {'type': 'route_block', 'delay_days': 6, 'needed_by_days': 4, 'disrupted_supplier': None, 'blocked_route': ...
    [MonitorAgent] tool extract_items_and_sites(report_text=ROUTE BLOCKED: Western Roadways repor...) -> {'items': ['ITM-007'], 'sites': ['WH-MUM'], 'unknown_items': [], 'unknown_sites': []}
[MonitorAgent] posted 'disruption' to blackboard (route_block reported; delay 6d, needed by 4d)
[MonitorAgent] posted 'affected_items' to blackboard (ITM-007)
[MonitorAgent] posted 'affected_sites' to blackboard (WH-MUM)
-- step 2: InventoryAgent | SupplierAgent | RouteAgent (parallel) --
    [InventoryAgent] tool get_stock_level(item_id=ITM-007, site_id=

## Scenario 6 - Minor delay, healthy stock (expected: plain hold)

In [7]:
r6 = await run_disruption(REPORTS_DIR / "06_false_alarm_hold.json", verbose=False)
print(r6.trace[-1])

>>> DECISION for ITM-002 @ WH-BLR: HOLD
    Rationale: HOLD at $0, the cheapest of 2 viable option(s) in cost_options:ITM-002@WH-BLR (CostAgent). Evidence: disruption (MonitorAgent) = supplier_delay, delay 2d, needed by 10d; inventory:ITM-002@WH-BLR (InventoryAgent) = 400 on hand vs safety 150, 26.7d cover; chosen because stock 400 vs safety 150, cover 26.7d vs delay 2d. Also viable but costlier: reorder SUP-04 $1,715. Rejected as not viable: reroute RT-02 $358 (not applicable: disruption is not on a route).
    Notify: Procurement Desk <procurement@voltride.example>; Warehouse Manager, Bengaluru <wm.bengaluru@voltride.example>


## Final blackboard of Scenario 1 (every posting with author, status and timestamp)

In [8]:
for p in r1.blackboard:
    print(f"{p['timestamp']}  {p['author']:<16} {p['status']:<7} {p['key']}")
    print("    value:", json.dumps(p['value'])[:300])
    if p['note']:
        print("    note :", p['note'][:300])

2026-11-11T14:26:47.681+00:00  MonitorAgent     ok      disruption
    value: {"type": "supplier_delay", "delay_days": 10, "needed_by_days": 7, "disrupted_supplier": "SUP-01", "blocked_route": null, "report_id": "DR-2026-1101"}
    note : supplier_delay reported; delay 10d, needed by 7d
2026-11-11T14:26:47.682+00:00  MonitorAgent     ok      affected_items
    value: {"items": ["ITM-001"], "unknown": []}
    note : ITM-001
2026-11-11T14:26:47.682+00:00  MonitorAgent     ok      affected_sites
    value: {"sites": ["WH-DEL"], "unknown": []}
    note : WH-DEL
2026-11-11T14:26:47.694+00:00  InventoryAgent   ok      inventory:ITM-001@WH-DEL
    value: {"item_id": "ITM-001", "site_id": "WH-DEL", "on_hand": 90, "safety_stock": 120, "daily_demand": 20, "days_of_cover": 4.5, "below_safety": true}
    note : 90 on hand, BELOW safety 120, covers 4.5d
2026-11-11T14:26:47.707+00:00  SupplierAgent    ok      supplier_options:ITM-001
    value: {"item_id": "ITM-001", "primary_supplier": "SUP-01", "p

## Options the Coordinator considered in Scenario 5

In [9]:
d = r5.decisions[0]
for o in d["options_considered"]:
    name = o["option"] + " " + (o.get("supplier_id") or o.get("route_id") or "")
    print(f"{name:<16} cost=${o['cost']:>9,.2f}  viable={str(o['viable']):<5}  {o['reason']}")
print("\nChosen:", d["action"], d["chosen_option"].get("route_id") or d["chosen_option"].get("supplier_id"))

hold             cost=$   840.00  viable=False  stock 100 vs safety 160, cover 2.5d vs delay 6d
reorder SUP-05   cost=$ 3,450.00  viable=True   SUP-05 lead time 3d vs needed-by 4d
reroute RT-06    cost=$   870.00  viable=True   RT-06 arrives in 2d vs needed-by 4d

Chosen: reroute RT-06


## Summary of all runs

In [10]:
for r in (r1, r2, r3, r4, r5, r6):
    d = r.decisions[0]
    print(f"{r.report_id}  {d['item_id']}@{d['site_id']}  ->  {r.action:<18} notify: {', '.join(n['role'] for n in d['notify'])}")

DR-2026-1101  ITM-001@WH-DEL  ->  reorder            notify: procurement, suppliers:SUP-03, suppliers:SUP-01, warehouse_managers:WH-DEL
DR-2026-1102  ITM-004@WH-MUM  ->  reorder            notify: procurement, suppliers:SUP-04, warehouse_managers:WH-MUM, customer_service
DR-2026-1103  ITM-006@WH-KOL  ->  reroute            notify: logistics_partners:BlueDart Air Cargo, warehouse_managers:WH-KOL
DR-2026-1104  ITM-008@WH-BLR  ->  hold + escalate    notify: escalation, procurement, customer_service, warehouse_managers:WH-BLR
DR-2026-1105  ITM-007@WH-MUM  ->  reroute            notify: logistics_partners:Konkan Rail Freight, warehouse_managers:WH-MUM
DR-2026-1106  ITM-002@WH-BLR  ->  hold               notify: procurement, warehouse_managers:WH-BLR


## Last lines of the append-only run log (`run_log.jsonl`)

In [11]:
with open("run_log.jsonl", encoding="utf-8") as fh:   # read-only view
    lines = fh.readlines()
print(f"{len(lines)} lines in run_log.jsonl; last 5:")
for line in lines[-5:]:
    print(line.rstrip()[:200])

483 lines in run_log.jsonl; last 5:
{"timestamp": "2026-11-11T14:26:48.102+00:00", "run_id": "20261111-195648-f7ef8a", "agent": "CoordinatorAgent", "action": "tool_call", "tool": "check_required_signals", "result_summary": "{'complete':
{"timestamp": "2026-11-11T14:26:48.103+00:00", "run_id": "20261111-195648-f7ef8a", "agent": "CoordinatorAgent", "action": "tool_call", "tool": "assess_options", "result_summary": "[{'option': 'hold', 
{"timestamp": "2026-11-11T14:26:48.105+00:00", "run_id": "20261111-195648-f7ef8a", "agent": "CoordinatorAgent", "action": "tool_call", "tool": "lookup_contacts", "result_summary": "[{'role': 'procurem
{"timestamp": "2026-11-11T14:26:48.108+00:00", "run_id": "20261111-195648-f7ef8a", "agent": "CoordinatorAgent", "action": "post:ok", "tool": "check_required_signals,assess_options,lookup_contacts", "r
{"timestamp": "2026-11-11T14:26:48.110+00:00", "run_id": "20261111-195648-f7ef8a", "agent": "Orchestrator", "action": "run_end", "tool": "", "result_summary": "